# 22. Robotics action policies — architecture-faithful, small-tensor implementations

Only tensor widths, batch size, image resolution, token count, action dimension/horizon, and training budget are reduced.

The implemented topology/algorithm is kept:

- **ACT**: ResNet-18 `[2,2,2,2]`, fixed posterior sinusoidal positions, 4-layer CVAE encoder, DETR-style 2D image positions, 4-layer observation encoder, 7-layer action-query decoder.
- **Diffusion Policy**: ResNet-18 observation encoder with GroupNorm, the original three-resolution conditional 1D U-Net topology, two residual blocks per stage, two middle blocks, FiLM scale/bias conditioning, and a 100-step stochastic DDPM schedule.
- **π0**: 27-layer SigLIP-style vision tower, 18-layer separate VLM/action-expert joint stack, GQA, RoPE, exact prefix/state/action attention blocks, Beta time sampling, flow matching, Euler rollout.
- **FAST**: quantile normalization -> DCT -> quantization -> frequency-major flattening -> actual BPE merge encode/decode -> inverse DCT.


In [ ]:
import math
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F


torch.manual_seed(12)
torch.set_num_threads(min(2, torch.get_num_threads()))
device = torch.device("cpu")
print("device:", device)


## 1. ACT — ResNet-18 + CVAE + DETR action queries

ACT uses a four-layer posterior Transformer over `[CLS, qpos, action chunk]`, then a four-layer observation encoder and seven-layer action-query decoder. Posterior positions are fixed sinusoidal values. Image features carry DETR-style two-dimensional sine/cosine positions rather than zero stand-ins.


In [ ]:
def fixed_sinusoidal_1d(length, dim, device):
    position = torch.arange(length, device=device, dtype=torch.float32)[:, None]
    frequencies = torch.exp(
        torch.arange(0, dim, 2, device=device, dtype=torch.float32)
        * (-math.log(10000.0) / dim)
    )
    angles = position * frequencies[None]

    encoding = torch.zeros(length, dim, device=device)
    encoding[:, 0::2] = angles.sin()
    encoding[:, 1::2] = angles.cos()
    return encoding


def detr_position_encoding_2d(batch, height, width, dim, device):
    assert dim % 4 == 0
    quarter = dim // 4

    y = torch.linspace(0.0, 1.0, height, device=device)
    x = torch.linspace(0.0, 1.0, width, device=device)
    grid_y, grid_x = torch.meshgrid(y, x, indexing="ij")

    frequencies = torch.exp(
        torch.arange(quarter, device=device, dtype=torch.float32)
        * (-math.log(10000.0) / max(quarter - 1, 1))
    )

    x_angle = grid_x[..., None] * frequencies
    y_angle = grid_y[..., None] * frequencies
    position = torch.cat(
        [x_angle.sin(), x_angle.cos(), y_angle.sin(), y_angle.cos()],
        dim=-1,
    )
    position = position.view(1, height * width, dim)
    return position.expand(batch, -1, -1)


class ResNetBasicBlock(nn.Module):
    def __init__(self, input_channels, output_channels, stride=1, norm="batch"):
        super().__init__()

        def make_norm(channels):
            if norm == "batch":
                return nn.BatchNorm2d(channels)
            if norm == "group":
                groups = min(8, channels)
                while channels % groups != 0:
                    groups -= 1
                return nn.GroupNorm(groups, channels)
            raise ValueError(norm)

        self.conv1 = nn.Conv2d(
            input_channels,
            output_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.norm1 = make_norm(output_channels)
        self.conv2 = nn.Conv2d(
            output_channels,
            output_channels,
            kernel_size=3,
            padding=1,
            bias=False,
        )
        self.norm2 = make_norm(output_channels)

        if stride == 1 and input_channels == output_channels:
            self.skip = nn.Identity()
        else:
            self.skip = nn.Sequential(
                nn.Conv2d(
                    input_channels,
                    output_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),
                make_norm(output_channels),
            )

    def forward(self, x):
        hidden = F.relu(self.norm1(self.conv1(x)))
        hidden = self.norm2(self.conv2(hidden))
        return F.relu(hidden + self.skip(x))


class SmallWidthResNet18(nn.Module):
    def __init__(self, output_channels=32, norm="batch"):
        super().__init__()

        def stem_norm(channels):
            if norm == "batch":
                return nn.BatchNorm2d(channels)
            groups = min(8, channels)
            while channels % groups != 0:
                groups -= 1
            return nn.GroupNorm(groups, channels)

        stage_widths = [8, 12, 16, 24]
        self.stem = nn.Sequential(
            nn.Conv2d(3, stage_widths[0], 7, stride=2, padding=3, bias=False),
            stem_norm(stage_widths[0]),
            nn.ReLU(),
            nn.MaxPool2d(3, stride=2, padding=1),
        )

        stages = []
        input_channels = stage_widths[0]
        for stage_index, stage_width in enumerate(stage_widths):
            stride = 1 if stage_index == 0 else 2
            stages.append(
                nn.Sequential(
                    ResNetBasicBlock(
                        input_channels,
                        stage_width,
                        stride=stride,
                        norm=norm,
                    ),
                    ResNetBasicBlock(
                        stage_width,
                        stage_width,
                        norm=norm,
                    ),
                )
            )
            input_channels = stage_width

        self.stages = nn.ModuleList(stages)
        self.projection = nn.Conv2d(input_channels, output_channels, 1)

    def forward(self, image):
        hidden = self.stem(image)
        for stage in self.stages:
            hidden = stage(hidden)
        return self.projection(hidden)


class SmallTensorACT(nn.Module):
    def __init__(
        self,
        action_dim=3,
        hidden_dim=32,
        chunk_size=4,
        latent_dim=8,
        heads=8,
    ):
        super().__init__()
        self.action_dim = action_dim
        self.chunk_size = chunk_size
        self.latent_dim = latent_dim

        self.backbone = SmallWidthResNet18(hidden_dim, norm="batch")

        self.posterior_cls = nn.Parameter(torch.zeros(1, 1, hidden_dim))
        self.posterior_qpos = nn.Linear(action_dim, hidden_dim)
        self.posterior_action = nn.Linear(action_dim, hidden_dim)

        posterior_layer = nn.TransformerEncoderLayer(
            hidden_dim,
            heads,
            2 * hidden_dim,
            batch_first=True,
        )
        self.posterior_encoder = nn.TransformerEncoder(
            posterior_layer,
            num_layers=4,
        )
        self.mu = nn.Linear(hidden_dim, latent_dim)
        self.logvar = nn.Linear(hidden_dim, latent_dim)

        self.latent_projection = nn.Linear(latent_dim, hidden_dim)
        self.qpos_projection = nn.Linear(action_dim, hidden_dim)
        self.additional_position = nn.Parameter(
            torch.randn(1, 2, hidden_dim) * 0.02
        )

        observation_layer = nn.TransformerEncoderLayer(
            hidden_dim,
            heads,
            2 * hidden_dim,
            batch_first=True,
        )
        self.observation_encoder = nn.TransformerEncoder(
            observation_layer,
            num_layers=4,
        )

        decoder_layer = nn.TransformerDecoderLayer(
            hidden_dim,
            heads,
            2 * hidden_dim,
            batch_first=True,
        )
        self.action_decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=7,
        )
        self.action_queries = nn.Parameter(
            torch.randn(1, chunk_size, hidden_dim) * 0.02
        )
        self.action_head = nn.Linear(hidden_dim, action_dim)

    def encode_posterior(self, qpos, target_actions):
        batch_size = qpos.size(0)
        tokens = torch.cat(
            [
                self.posterior_cls.expand(batch_size, -1, -1),
                self.posterior_qpos(qpos).unsqueeze(1),
                self.posterior_action(target_actions),
            ],
            dim=1,
        )
        position = fixed_sinusoidal_1d(
            tokens.size(1),
            tokens.size(2),
            tokens.device,
        )[None]
        posterior_hidden = self.posterior_encoder(tokens + position)[:, 0]

        mu = self.mu(posterior_hidden)
        logvar = self.logvar(posterior_hidden)
        latent = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        return latent, mu, logvar

    def forward(self, image, qpos, target_actions=None):
        batch_size = image.size(0)
        if target_actions is None:
            latent = torch.zeros(
                batch_size,
                self.latent_dim,
                device=image.device,
            )
            mu = torch.zeros_like(latent)
            logvar = torch.zeros_like(latent)
        else:
            latent, mu, logvar = self.encode_posterior(
                qpos,
                target_actions,
            )

        image_features = self.backbone(image)
        _, _, feature_height, feature_width = image_features.shape
        image_tokens = image_features.flatten(2).transpose(1, 2)
        image_position = detr_position_encoding_2d(
            batch_size,
            feature_height,
            feature_width,
            image_tokens.size(-1),
            image.device,
        )

        memory = torch.cat(
            [
                self.latent_projection(latent).unsqueeze(1),
                self.qpos_projection(qpos).unsqueeze(1),
                image_tokens,
            ],
            dim=1,
        )
        memory_position = torch.cat(
            [
                self.additional_position.expand(batch_size, -1, -1),
                image_position,
            ],
            dim=1,
        )
        memory = self.observation_encoder(memory + memory_position)

        queries = self.action_queries.expand(batch_size, -1, -1)
        decoded = self.action_decoder(queries, memory)
        return self.action_head(decoded), mu, logvar


act = SmallTensorACT().to(device)
assert [len(stage) for stage in act.backbone.stages] == [2, 2, 2, 2]
assert len(act.posterior_encoder.layers) == 4
assert len(act.observation_encoder.layers) == 4
assert len(act.action_decoder.layers) == 7

act_image = torch.randn(2, 3, 32, 32, device=device)
act_qpos = torch.randn(2, 3, device=device)
act_target = torch.randn(2, 4, 3, device=device)

predicted, mu, logvar = act(act_image, act_qpos, act_target)
reconstruction = F.l1_loss(predicted, act_target)
kl = -0.5 * (1 + logvar - mu.square() - logvar.exp()).mean()
(reconstruction + 0.005 * kl).backward()
print("ACT output:", predicted.shape)


## 2. Diffusion Policy — ResNet observation encoder + original conditional 1D U-Net + DDPM

The U-Net follows the released `ConditionalUnet1D`: three down resolutions, two conditional residual blocks at each down stage, two middle blocks, two up transitions with two blocks each, and the final convolution block. The image-policy setting uses scale-and-bias FiLM. Sampling uses the full 100-step DDPM reverse chain rather than a deterministic cosine shortcut.


In [ ]:
def diffusion_time_embedding(timestep, dim):
    half = dim // 2
    frequencies = torch.exp(
        -math.log(10000.0)
        * torch.arange(half, device=timestep.device, dtype=torch.float32)
        / max(half - 1, 1)
    )
    angles = timestep.float()[:, None] * frequencies[None]
    return torch.cat([angles.sin(), angles.cos()], dim=-1)


class Conv1dBlock(nn.Module):
    def __init__(self, input_channels, output_channels, kernel_size=5):
        super().__init__()
        groups = min(8, output_channels)
        while output_channels % groups != 0:
            groups -= 1
        self.conv = nn.Conv1d(
            input_channels,
            output_channels,
            kernel_size,
            padding=kernel_size // 2,
        )
        self.norm = nn.GroupNorm(groups, output_channels)

    def forward(self, x):
        return F.mish(self.norm(self.conv(x)))


class ConditionalResidualBlock1D(nn.Module):
    def __init__(
        self,
        input_channels,
        output_channels,
        condition_dim,
        kernel_size=5,
    ):
        super().__init__()
        self.output_channels = output_channels
        self.block1 = Conv1dBlock(
            input_channels,
            output_channels,
            kernel_size,
        )
        self.block2 = Conv1dBlock(
            output_channels,
            output_channels,
            kernel_size,
        )
        self.condition_encoder = nn.Sequential(
            nn.Mish(),
            nn.Linear(condition_dim, 2 * output_channels),
        )
        self.residual = (
            nn.Identity()
            if input_channels == output_channels
            else nn.Conv1d(input_channels, output_channels, 1)
        )

    def forward(self, x, condition):
        hidden = self.block1(x)
        modulation = self.condition_encoder(condition)
        scale, bias = modulation.view(
            modulation.size(0),
            2,
            self.output_channels,
            1,
        ).unbind(dim=1)
        hidden = scale * hidden + bias
        hidden = self.block2(hidden)
        return hidden + self.residual(x)


class Downsample1D(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv1d(channels, channels, 3, stride=2, padding=1)

    def forward(self, x):
        return self.conv(x)


class Upsample1D(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.ConvTranspose1d(
            channels,
            channels,
            4,
            stride=2,
            padding=1,
        )

    def forward(self, x):
        return self.conv(x)


class ConditionalUnet1D(nn.Module):
    def __init__(
        self,
        action_dim=3,
        global_condition_dim=32,
        diffusion_embed_dim=32,
        down_dims=(8, 16, 32),
        kernel_size=5,
    ):
        super().__init__()
        self.down_dims = tuple(down_dims)
        self.diffusion_embed_dim = diffusion_embed_dim

        self.diffusion_step_encoder = nn.Sequential(
            nn.Linear(diffusion_embed_dim, 4 * diffusion_embed_dim),
            nn.Mish(),
            nn.Linear(4 * diffusion_embed_dim, diffusion_embed_dim),
        )
        condition_dim = diffusion_embed_dim + global_condition_dim

        all_dims = [action_dim] + list(down_dims)
        in_out = list(zip(all_dims[:-1], all_dims[1:]))

        self.down_modules = nn.ModuleList()
        for stage_index, (input_channels, output_channels) in enumerate(in_out):
            is_last = stage_index == len(in_out) - 1
            self.down_modules.append(
                nn.ModuleList(
                    [
                        ConditionalResidualBlock1D(
                            input_channels,
                            output_channels,
                            condition_dim,
                            kernel_size,
                        ),
                        ConditionalResidualBlock1D(
                            output_channels,
                            output_channels,
                            condition_dim,
                            kernel_size,
                        ),
                        nn.Identity()
                        if is_last
                        else Downsample1D(output_channels),
                    ]
                )
            )

        deepest = down_dims[-1]
        self.mid_modules = nn.ModuleList(
            [
                ConditionalResidualBlock1D(
                    deepest,
                    deepest,
                    condition_dim,
                    kernel_size,
                ),
                ConditionalResidualBlock1D(
                    deepest,
                    deepest,
                    condition_dim,
                    kernel_size,
                ),
            ]
        )

        self.up_modules = nn.ModuleList()
        reversed_pairs = list(reversed(in_out[1:]))
        for stage_index, (input_channels, output_channels) in enumerate(reversed_pairs):
            is_last = stage_index == len(reversed_pairs) - 1
            self.up_modules.append(
                nn.ModuleList(
                    [
                        ConditionalResidualBlock1D(
                            2 * output_channels,
                            input_channels,
                            condition_dim,
                            kernel_size,
                        ),
                        ConditionalResidualBlock1D(
                            input_channels,
                            input_channels,
                            condition_dim,
                            kernel_size,
                        ),
                        Upsample1D(input_channels),
                    ]
                )
            )

        self.final_conv = nn.Sequential(
            Conv1dBlock(down_dims[0], down_dims[0], kernel_size),
            nn.Conv1d(down_dims[0], action_dim, 1),
        )

    def forward(self, noisy_actions, timestep, global_condition):
        if timestep.ndim == 0:
            timestep = timestep.expand(noisy_actions.size(0))

        time_embedding = diffusion_time_embedding(
            timestep,
            self.diffusion_embed_dim,
        )
        time_embedding = self.diffusion_step_encoder(time_embedding)
        condition = torch.cat([time_embedding, global_condition], dim=-1)

        hidden = noisy_actions.transpose(1, 2)
        skips = []
        for block1, block2, downsample in self.down_modules:
            hidden = block1(hidden, condition)
            hidden = block2(hidden, condition)
            skips.append(hidden)
            hidden = downsample(hidden)

        for middle in self.mid_modules:
            hidden = middle(hidden, condition)

        for block1, block2, upsample in self.up_modules:
            skip = skips.pop()
            if hidden.size(-1) != skip.size(-1):
                hidden = F.interpolate(
                    hidden,
                    size=skip.size(-1),
                    mode="linear",
                    align_corners=False,
                )
            hidden = torch.cat([hidden, skip], dim=1)
            hidden = block1(hidden, condition)
            hidden = block2(hidden, condition)
            hidden = upsample(hidden)

        return self.final_conv(hidden).transpose(1, 2)


def squared_cosine_betas(num_steps=100, max_beta=0.999):
    def alpha_bar(time):
        return math.cos((time + 0.008) / 1.008 * math.pi / 2) ** 2

    betas = []
    for step in range(num_steps):
        t1 = step / num_steps
        t2 = (step + 1) / num_steps
        beta = 1 - alpha_bar(t2) / alpha_bar(t1)
        betas.append(min(beta, max_beta))
    return torch.tensor(betas, dtype=torch.float32)


class SmallScheduleDDPMScheduler:
    def __init__(self, num_train_timesteps=100):
        self.num_train_timesteps = num_train_timesteps
        self.betas = squared_cosine_betas(num_train_timesteps)
        self.alphas = 1.0 - self.betas
        self.alpha_bars = torch.cumprod(self.alphas, dim=0)

    def add_noise(self, clean, noise, timestep):
        alpha_bar = self.alpha_bars.to(clean.device)[timestep]
        alpha_bar = alpha_bar[:, None, None]
        return (
            alpha_bar.sqrt() * clean
            + (1 - alpha_bar).sqrt() * noise
        )

    def step(self, predicted_noise, timestep, sample):
        betas = self.betas.to(sample.device)
        alphas = self.alphas.to(sample.device)
        alpha_bars = self.alpha_bars.to(sample.device)

        alpha_t = alphas[timestep]
        beta_t = betas[timestep]
        alpha_bar_t = alpha_bars[timestep]
        if timestep > 0:
            alpha_bar_previous = alpha_bars[timestep - 1]
        else:
            alpha_bar_previous = sample.new_tensor(1.0)

        x0_prediction = (
            sample
            - torch.sqrt(1 - alpha_bar_t) * predicted_noise
        ) / torch.sqrt(alpha_bar_t)
        x0_prediction = x0_prediction.clamp(-1.0, 1.0)

        x0_coefficient = (
            torch.sqrt(alpha_bar_previous)
            * beta_t
            / (1 - alpha_bar_t)
        )
        xt_coefficient = (
            torch.sqrt(alpha_t)
            * (1 - alpha_bar_previous)
            / (1 - alpha_bar_t)
        )
        mean = x0_coefficient * x0_prediction + xt_coefficient * sample

        if timestep == 0:
            return mean

        variance = (
            beta_t
            * (1 - alpha_bar_previous)
            / (1 - alpha_bar_t)
        )
        return mean + variance.clamp_min(1e-20).sqrt() * torch.randn_like(sample)


class SmallTensorDiffusionPolicy(nn.Module):
    def __init__(self, action_dim=3, condition_dim=32):
        super().__init__()
        self.observation_encoder = SmallWidthResNet18(
            condition_dim,
            norm="group",
        )
        self.model = ConditionalUnet1D(
            action_dim=action_dim,
            global_condition_dim=condition_dim,
            down_dims=(8, 16, 32),
            kernel_size=5,
        )

    def encode_observation(self, image):
        feature_map = self.observation_encoder(image)
        return feature_map.mean(dim=(-2, -1))

    def forward(self, noisy_actions, image, timestep):
        condition = self.encode_observation(image)
        return self.model(noisy_actions, timestep, condition)


@torch.no_grad()
def sample_diffusion_policy(model, scheduler, image, horizon=8, action_dim=3):
    actions = torch.randn(image.size(0), horizon, action_dim, device=image.device)
    for timestep in reversed(range(scheduler.num_train_timesteps)):
        time_batch = torch.full(
            (image.size(0),),
            timestep,
            dtype=torch.long,
            device=image.device,
        )
        predicted_noise = model(actions, image, time_batch)
        actions = scheduler.step(predicted_noise, timestep, actions)
    return actions


diffusion_policy = SmallTensorDiffusionPolicy().to(device)
scheduler = SmallScheduleDDPMScheduler(num_train_timesteps=100)

assert len(diffusion_policy.observation_encoder.stages) == 4
assert [len(stage) for stage in diffusion_policy.observation_encoder.stages] == [2, 2, 2, 2]
assert len(diffusion_policy.model.down_modules) == 3
assert all(len(stage) == 3 for stage in diffusion_policy.model.down_modules)
assert len(diffusion_policy.model.mid_modules) == 2
assert len(diffusion_policy.model.up_modules) == 2
assert scheduler.num_train_timesteps == 100

policy_image = torch.randn(1, 3, 32, 32, device=device)
clean_actions = torch.randn(1, 8, 3, device=device)
noise = torch.randn_like(clean_actions)
timestep = torch.tensor([37], device=device)
noisy_actions = scheduler.add_noise(clean_actions, noise, timestep)
predicted_noise = diffusion_policy(noisy_actions, policy_image, timestep)
F.mse_loss(predicted_noise, noise).backward()

sampled_actions = sample_diffusion_policy(
    diffusion_policy,
    scheduler,
    policy_image,
)
print("Diffusion Policy rollout:", sampled_actions.shape)


## 3. π0 — 27-layer vision tower + 18-layer joint VLM/action-expert stack

The vision and language prefix is one full-attention block. The robot state starts a new block, so it may read the prefix but not future action tokens. The first action token starts the action block and all action tokens share that block. VLM and action expert keep separate parameters while Q/K/V participate in one joint attention calculation.


In [ ]:
def apply_rope(x, positions, base=10000.0):
    head_dim = x.size(-1)
    assert head_dim % 2 == 0
    pair_index = torch.arange(
        0,
        head_dim,
        2,
        device=x.device,
        dtype=torch.float32,
    )
    inverse_frequency = 1.0 / (base ** (pair_index / head_dim))
    angles = positions.float()[None, None, :, None] * inverse_frequency[None, None, None, :]

    even = x[..., 0::2]
    odd = x[..., 1::2]
    rotated_even = even * angles.cos() - odd * angles.sin()
    rotated_odd = even * angles.sin() + odd * angles.cos()
    return torch.stack([rotated_even, rotated_odd], dim=-1).flatten(-2)


def repeat_kv(x, query_heads):
    if x.size(1) == query_heads:
        return x
    assert query_heads % x.size(1) == 0
    repeat_factor = query_heads // x.size(1)
    return x.repeat_interleave(repeat_factor, dim=1)


def block_causal_mask(block_starts):
    block_ids = torch.cumsum(block_starts.long(), dim=-1)
    query_block = block_ids[:, :, None]
    key_block = block_ids[:, None, :]
    return key_block <= query_block


def pi0_time_embedding(t, dim, min_period=4e-3, max_period=4.0):
    half = dim // 2
    periods = torch.exp(
        torch.linspace(
            math.log(min_period),
            math.log(max_period),
            half,
            device=t.device,
        )
    )
    angles = t[:, None] * (2 * math.pi / periods)[None]
    return torch.cat([angles.sin(), angles.cos()], dim=-1)


class VisionTransformerBlock(nn.Module):
    def __init__(self, dim=32, heads=16):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attention = nn.MultiheadAttention(
            dim,
            heads,
            batch_first=True,
        )
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.GELU(),
            nn.Linear(4 * dim, dim),
        )

    def forward(self, x):
        hidden = self.norm1(x)
        attended, _ = self.attention(
            hidden,
            hidden,
            hidden,
            need_weights=False,
        )
        x = x + attended
        return x + self.mlp(self.norm2(x))


class SmallWidthSigLIPVision(nn.Module):
    def __init__(self, dim=32, depth=27, heads=16, patch_size=14):
        super().__init__()
        self.patch_size = patch_size
        self.patch = nn.Conv2d(
            3,
            dim,
            patch_size,
            stride=patch_size,
        )
        self.blocks = nn.ModuleList(
            [VisionTransformerBlock(dim, heads) for _ in range(depth)]
        )
        self.norm = nn.LayerNorm(dim)

    def forward(self, image):
        tokens = self.patch(image).flatten(2).transpose(1, 2)
        for block in self.blocks:
            tokens = block(tokens)
        return self.norm(tokens)


class Pi0StreamParameters(nn.Module):
    def __init__(self, dim=32, query_heads=8, kv_heads=1):
        super().__init__()
        assert dim % query_heads == 0
        self.dim = dim
        self.query_heads = query_heads
        self.kv_heads = kv_heads
        self.head_dim = dim // query_heads

        self.attention_norm = nn.RMSNorm(dim)
        self.q_proj = nn.Linear(dim, query_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(dim, kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(dim, kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(query_heads * self.head_dim, dim, bias=False)

        self.ffn_norm = nn.RMSNorm(dim)
        self.gate_proj = nn.Linear(dim, 4 * dim, bias=False)
        self.value_proj = nn.Linear(dim, 4 * dim, bias=False)
        self.ffn_out = nn.Linear(4 * dim, dim, bias=False)

    def project_qkv(self, x, positions):
        batch, length, _ = x.shape
        hidden = self.attention_norm(x)

        q = self.q_proj(hidden).view(
            batch,
            length,
            self.query_heads,
            self.head_dim,
        ).transpose(1, 2)
        k = self.k_proj(hidden).view(
            batch,
            length,
            self.kv_heads,
            self.head_dim,
        ).transpose(1, 2)
        v = self.v_proj(hidden).view(
            batch,
            length,
            self.kv_heads,
            self.head_dim,
        ).transpose(1, 2)

        q = apply_rope(q, positions)
        k = apply_rope(k, positions)
        return q, k, v

    def ffn(self, x):
        hidden = self.ffn_norm(x)
        gate = F.gelu(self.gate_proj(hidden), approximate="tanh")
        value = self.value_proj(hidden)
        return self.ffn_out(gate * value)


class Pi0JointBlock(nn.Module):
    def __init__(self, dim=32, query_heads=8, kv_heads=1):
        super().__init__()
        self.query_heads = query_heads
        self.base = Pi0StreamParameters(dim, query_heads, kv_heads)
        self.expert = Pi0StreamParameters(dim, query_heads, kv_heads)

    def forward(self, prefix, suffix, attention_mask):
        prefix_positions = torch.arange(prefix.size(1), device=prefix.device)
        suffix_positions = torch.arange(
            prefix.size(1),
            prefix.size(1) + suffix.size(1),
            device=prefix.device,
        )

        prefix_q, prefix_k, prefix_v = self.base.project_qkv(
            prefix,
            prefix_positions,
        )
        suffix_q, suffix_k, suffix_v = self.expert.project_qkv(
            suffix,
            suffix_positions,
        )

        q = torch.cat([prefix_q, suffix_q], dim=2)
        k = torch.cat(
            [
                repeat_kv(prefix_k, self.query_heads),
                repeat_kv(suffix_k, self.query_heads),
            ],
            dim=2,
        )
        v = torch.cat(
            [
                repeat_kv(prefix_v, self.query_heads),
                repeat_kv(suffix_v, self.query_heads),
            ],
            dim=2,
        )

        attended = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attention_mask[:, None],
        )
        prefix_length = prefix.size(1)
        prefix_attended = attended[:, :, :prefix_length]
        suffix_attended = attended[:, :, prefix_length:]

        prefix_attended = prefix_attended.transpose(1, 2).contiguous().flatten(2)
        suffix_attended = suffix_attended.transpose(1, 2).contiguous().flatten(2)

        prefix = prefix + self.base.o_proj(prefix_attended)
        suffix = suffix + self.expert.o_proj(suffix_attended)
        prefix = prefix + self.base.ffn(prefix)
        suffix = suffix + self.expert.ffn(suffix)
        return prefix, suffix


class SmallTensorPi0(nn.Module):
    def __init__(
        self,
        dim=32,
        vocab=64,
        action_dim=3,
        horizon=4,
        vision_depth=27,
        joint_depth=18,
    ):
        super().__init__()
        self.dim = dim
        self.action_dim = action_dim
        self.horizon = horizon

        self.vision = SmallWidthSigLIPVision(
            dim=dim,
            depth=vision_depth,
            heads=16,
            patch_size=14,
        )
        self.language = nn.Embedding(vocab, dim)
        self.state_projection = nn.Linear(action_dim, dim)
        self.action_projection = nn.Linear(action_dim, dim)
        self.action_time_mlp = nn.Sequential(
            nn.Linear(2 * dim, dim),
            nn.SiLU(),
            nn.Linear(dim, dim),
        )

        self.blocks = nn.ModuleList(
            [Pi0JointBlock(dim, query_heads=8, kv_heads=1) for _ in range(joint_depth)]
        )
        self.velocity = nn.Linear(dim, action_dim)

    def build_tokens(self, image, language, state, noisy_actions, t):
        image_tokens = self.vision(image)
        language_tokens = self.language(language)
        prefix = torch.cat([image_tokens, language_tokens], dim=1)

        state_token = self.state_projection(state).unsqueeze(1)
        action_tokens = self.action_projection(noisy_actions)
        time_embedding = pi0_time_embedding(t, self.dim)
        expanded_time = time_embedding[:, None, :].expand(
            -1,
            action_tokens.size(1),
            -1,
        )
        action_tokens = self.action_time_mlp(
            torch.cat([action_tokens, expanded_time], dim=-1)
        )
        suffix = torch.cat([state_token, action_tokens], dim=1)

        prefix_starts = torch.zeros(
            prefix.size(0),
            prefix.size(1),
            dtype=torch.bool,
            device=prefix.device,
        )
        prefix_starts[:, 0] = True

        suffix_starts = torch.zeros(
            suffix.size(0),
            suffix.size(1),
            dtype=torch.bool,
            device=suffix.device,
        )
        suffix_starts[:, 0] = True      # state block
        suffix_starts[:, 1] = True      # action block starts at first action token

        block_starts = torch.cat([prefix_starts, suffix_starts], dim=1)
        attention_mask = block_causal_mask(block_starts)
        return prefix, suffix, attention_mask

    def forward(self, image, language, state, noisy_actions, t):
        prefix, suffix, attention_mask = self.build_tokens(
            image,
            language,
            state,
            noisy_actions,
            t,
        )
        for block in self.blocks:
            prefix, suffix = block(prefix, suffix, attention_mask)
        return self.velocity(suffix[:, 1:])


@torch.no_grad()
def sample_pi0(model, image, language, state, steps=6):
    actions = torch.randn(
        state.size(0),
        model.horizon,
        model.action_dim,
        device=state.device,
    )
    delta = 1.0 / steps
    for step in range(steps, 0, -1):
        t = torch.full(
            (state.size(0),),
            step / steps,
            device=state.device,
        )
        velocity = model(image, language, state, actions, t)
        actions = actions - delta * velocity
    return actions


pi0 = SmallTensorPi0().to(device)
assert len(pi0.vision.blocks) == 27
assert len(pi0.blocks) == 18
assert pi0.blocks[0].base.query_heads == 8
assert pi0.blocks[0].base.kv_heads == 1

pi0_image = torch.randn(1, 3, 28, 28, device=device)
pi0_language = torch.randint(0, 64, (1, 3), device=device)
pi0_state = torch.randn(1, 3, device=device)
pi0_actions = torch.randn(1, 4, 3, device=device)
pi0_noise = torch.randn_like(pi0_actions)

beta_distribution = torch.distributions.Beta(1.5, 1.0)
t = beta_distribution.sample((1,)).to(device) * 0.999 + 0.001
noisy_actions = (
    t[:, None, None] * pi0_noise
    + (1 - t[:, None, None]) * pi0_actions
)
target_velocity = pi0_noise - pi0_actions
predicted_velocity = pi0(
    pi0_image,
    pi0_language,
    pi0_state,
    noisy_actions,
    t,
)
F.mse_loss(predicted_velocity, target_velocity).backward()

with torch.no_grad():
    _, _, mask = pi0.build_tokens(
        pi0_image,
        pi0_language,
        pi0_state,
        noisy_actions,
        t,
    )
    prefix_length = pi0.vision(pi0_image).size(1) + pi0_language.size(1)
    state_index = prefix_length
    first_action_index = prefix_length + 1
    assert not mask[0, state_index, first_action_index]
    assert mask[0, first_action_index, state_index]
    assert mask[0, -1, first_action_index]

print("pi0 rollout:", sample_pi0(pi0, pi0_image, pi0_language, pi0_state).shape)


## 4. FAST — DCT, quantization, BPE, inverse DCT

FAST first normalizes action coordinates with robust quantiles, applies DCT along time, quantizes coefficients, flattens in frequency-major order, and then applies BPE. The decoder expands the actual BPE token stream before inverse quantization and inverse DCT; it never reconstructs from a pre-BPE shortcut.


In [ ]:
def dct_matrix(length, device):
    time_index = torch.arange(length, device=device, dtype=torch.float32)
    frequency_index = torch.arange(
        length,
        device=device,
        dtype=torch.float32,
    )[:, None]
    matrix = torch.cos(
        math.pi / length
        * (time_index + 0.5)
        * frequency_index
    )
    matrix[0] *= math.sqrt(1.0 / length)
    matrix[1:] *= math.sqrt(2.0 / length)
    return matrix


def robust_bounds(calibration):
    flat = calibration.reshape(-1, calibration.size(-1))
    low = torch.quantile(flat, 0.01, dim=0)
    high = torch.quantile(flat, 0.99, dim=0)
    return low, high


def normalize_actions(actions, low, high):
    normalized = 2 * (actions - low) / (high - low).clamp_min(1e-6) - 1
    return normalized.clamp(-1.0, 1.0)


def denormalize_actions(actions, low, high):
    return 0.5 * (actions + 1) * (high - low) + low


def dct_actions(actions):
    matrix = dct_matrix(actions.size(-2), actions.device)
    return torch.einsum("ft,btd->bfd", matrix, actions)


def inverse_dct(coefficients):
    matrix = dct_matrix(coefficients.size(-2), coefficients.device)
    return torch.einsum("tf,bfd->btd", matrix.transpose(0, 1), coefficients)


def quantize_coefficients(coefficients, bins=256, clip=4.0):
    clipped = coefficients.clamp(-clip, clip)
    scaled = (clipped + clip) / (2 * clip)
    return torch.round(scaled * (bins - 1)).long()


def dequantize_coefficients(tokens, bins=256, clip=4.0):
    scaled = tokens.float() / (bins - 1)
    return scaled * (2 * clip) - clip


class SimpleBPE:
    def __init__(self):
        self.merge_rules = []
        self.expansions = {}
        self.next_token = 0

    def fit(self, sequences, merges=16):
        max_symbol = max(max(sequence) for sequence in sequences)
        self.next_token = max_symbol + 1
        working = [list(sequence) for sequence in sequences]

        for _ in range(merges):
            pair_counts = Counter()
            for sequence in working:
                pair_counts.update(zip(sequence[:-1], sequence[1:]))
            if not pair_counts:
                break

            pair, count = pair_counts.most_common(1)[0]
            if count < 2:
                break

            new_token = self.next_token
            self.next_token += 1
            self.merge_rules.append((pair, new_token))
            self.expansions[new_token] = pair
            working = [
                self._apply_single_merge(sequence, pair, new_token)
                for sequence in working
            ]

    @staticmethod
    def _apply_single_merge(sequence, pair, new_token):
        output = []
        index = 0
        while index < len(sequence):
            if (
                index + 1 < len(sequence)
                and sequence[index] == pair[0]
                and sequence[index + 1] == pair[1]
            ):
                output.append(new_token)
                index += 2
            else:
                output.append(sequence[index])
                index += 1
        return output

    def encode(self, sequence):
        encoded = list(sequence)
        for pair, new_token in self.merge_rules:
            encoded = self._apply_single_merge(encoded, pair, new_token)
        return encoded

    def _expand_token(self, token):
        if token not in self.expansions:
            return [token]
        left, right = self.expansions[token]
        return self._expand_token(left) + self._expand_token(right)

    def decode(self, encoded):
        decoded = []
        for token in encoded:
            decoded.extend(self._expand_token(token))
        return decoded


def flatten_frequency_major(quantized):
    return quantized.transpose(1, 2).reshape(quantized.size(0), -1)


def unflatten_frequency_major(flattened, horizon, action_dim):
    return flattened.view(flattened.size(0), action_dim, horizon).transpose(1, 2)


calibration = torch.randn(32, 8, 3, device=device)
low, high = robust_bounds(calibration)
normalized = normalize_actions(calibration, low, high)
coefficients = dct_actions(normalized)
quantized = quantize_coefficients(coefficients)
flat = flatten_frequency_major(quantized)

bpe = SimpleBPE()
bpe.fit([row.tolist() for row in flat], merges=16)
encoded = [bpe.encode(row.tolist()) for row in flat]
decoded = [bpe.decode(sequence) for sequence in encoded]

decoded_tensor = torch.tensor(decoded, device=device, dtype=torch.long)
assert torch.equal(decoded_tensor, flat)

restored_quantized = unflatten_frequency_major(
    decoded_tensor,
    horizon=8,
    action_dim=3,
)
restored_coefficients = dequantize_coefficients(restored_quantized)
restored_normalized = inverse_dct(restored_coefficients)
restored_actions = denormalize_actions(restored_normalized, low, high)

print("FAST BPE merges:", len(bpe.merge_rules))
print("FAST encoded lengths:", [len(sequence) for sequence in encoded[:4]])
print("FAST restored shape:", restored_actions.shape)


## Structural checklist

The assertions above verify ACT `[2,2,2,2] + 4/4/7`, Diffusion Policy's three down resolutions/two middle/two up transitions and 100 DDPM steps, and π0's 27 vision + 18 joint layers with GQA. The attention-mask assertions explicitly verify the prefix/state/action information-flow rule. FAST verifies exact BPE round-trip equality before inverse DCT.
